## Fine-Tunning of station status parameters

In this notebook, we analyze changes in status behaviours varying station parameters of a group of selected stations.

### Module imports

In this section, we load checklist modules to be able to use in main script file. It's required to run only one time bellow cell due to every new execution change current execution path. In other case it's required to restart jupyter variables.

In [ ]:
## Trick to change current location
%cd ..
import numpy as np
import pandas as pd
import time
from datetime import datetime, UTC
from tqdm import tqdm
from src.config import settings
from src.data_access.slinktool_executor import SlinktoolExecutor
from src.data_access.load_data import StationRepository
from src.services.delay_calculator import DelayCalculator
from src.services.station_analysis import StationAnalysis
from src.utilities.data_formatter import SiigeoFormatter, OtherStationsFormatter
from src.utilities.slinktool_formatter import SlinktoolFormatter

/home/wpenaranda/error_404/check_list


### Load station information from SGC

In [2]:
# Load stations data
siigeo_data = StationRepository.get_siigeo_stations()
shared_data = StationRepository.get_shared_stations()
inter_data = StationRepository.get_inter_stations()

data = {'siigeo': siigeo_data, 'shared': shared_data, 'inter': inter_data}
format_data = {}
for key, df in data.items():
    strategy = SiigeoFormatter() if key == 'siigeo' else OtherStationsFormatter()
    df = strategy.rename_cols(df)
    df = strategy.assign_data_types(df)
    df = strategy.infer_channel(df)
    df = strategy.filter_data(df)
    df = strategy.default_connection(df)
    format_data[key] = df     

siigeo_data = format_data['siigeo']
shared_data = format_data['shared']
inter_data = format_data['inter']
all_data = pd.concat([siigeo_data, shared_data, inter_data], ignore_index=True)

20:35:06 [INFO] Stations from Siigeo file succesfully load.
20:35:06 [INFO] Shared stations succesfully load.
20:35:06 [INFO] International stations succesfully load.
20:35:06 [INFO] Dataframe formatted succesfully. n=208
20:35:06 [INFO] Dataframe formatted succesfully. n=20
20:35:06 [INFO] Dataframe formatted succesfully. n=65


### Test time differences between validations.

Here, we test time differences between validations, for this case we propose following cases: (5 s, 10 s, 15 s, 30 s, 45 s, 60 s, 90 s) with a selection of a sample of stations to analyze any variations in delays and possible station status.

About stations, we select three different groups of stations: 

- stations that always are online in actual checklist.
- stations that usually have status from recovering to offline, but they usually recover information throughout the day. 
- stations that usually are offline.

According to actual date (03 August, 2026), we select these stations: first group: ARAMC, CHI, DBB and GUY2C; second group: BRJC, AGCC, POP2 and PPLP and thirth group: MEDEC, MOCC, GMALF and UPD2.

#### Time difference = 5 s

In [ ]:
# Request slinktool data 
port = SlinktoolExecutor.get_port('91')
comm = SlinktoolExecutor.build_command('91', port)
output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data = DelayCalculator.add_stations_untracked(all_data, slink_data)

# Compute Delay Time
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data = DelayCalculator.compute_delays(final_data, actual_time, 'last_connection')
final_data = DelayCalculator.format_delays(final_data, delay_column='delay')
print(f'to = {actual_time}')

# Second request
period_time = 5
with tqdm(total=period_time, desc=f"Waiting {period_time} seconds to execute a new validation") as pbar:
        for _ in range(period_time):
                time.sleep(1)
                pbar.update(1)

output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data2 = DelayCalculator.add_stations_untracked(all_data, slink_data)
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data2 = DelayCalculator.compute_delays(final_data2, actual_time, 'last_connection')
final_data2 = DelayCalculator.format_delays(final_data2, delay_column='delay')

# Extract delay columns
delay1 = final_data['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
delay2 = final_data2['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
status = StationAnalysis.get_station_status(delay1, delay2, period_time)
final_data2['delay1_seconds'] = delay1
final_data2['delay2_seconds'] = delay2
final_data2['status'] = status
print(f'to = {actual_time}')

# Subset stations of interest 
interm_sts = ['BRJC', 'POP2', 'AGCC', 'PPLP']
perma_sts = ['ARAMC', 'CHI', 'DBB', 'GUY2C']
off_sts = ['MEDEC', 'MOCC', 'GMALF', 'UPD2']
cols = ['stid', 'loc', 'channel', 'last_connection', 'delay1_seconds', 'delay2_seconds', 'status']
res = final_data2.loc[final_data2['stid'].isin([*perma_sts, *interm_sts, *off_sts])]
res = res[cols]
res.head(50)

20:36:26 [INFO] Port stablished succesfully.
20:36:26 [INFO] Sucessful information request from slinktool.
20:36:26 [INFO] Number of sensors to be processed: 1518
20:36:26 [INFO] Success!!
20:36:26 [INFO] Dataframe created succesfully!!
20:36:26 [INFO] Selection of the 792 field sensors completed succesfully!!
20:36:26 [INFO] Transmission type assigned succesfully!!
20:36:26 [INFO] Dataframe formatted succesfully according to SGC data!!
20:36:26 [INFO] Adding stations not tracked by slinktool succesfully!!
20:36:26 [INFO] Delays succesfully computed!!
20:36:26 [INFO] Delays succesfully formatted!!


to = 2026-08-04 01:36:26.180885


Waiting 5 seconds to execute a new validation: 100%|██████████| 5/5 [00:05<00:00,  1.00s/it]
20:36:31 [INFO] Sucessful information request from slinktool.
20:36:31 [INFO] Number of sensors to be processed: 1518
20:36:31 [INFO] Success!!
20:36:31 [INFO] Dataframe created succesfully!!
20:36:31 [INFO] Selection of the 792 field sensors completed succesfully!!
20:36:31 [INFO] Transmission type assigned succesfully!!
20:36:31 [INFO] Dataframe formatted succesfully according to SGC data!!
20:36:31 [INFO] Adding stations not tracked by slinktool succesfully!!
20:36:31 [INFO] Delays succesfully computed!!
20:36:31 [INFO] Delays succesfully formatted!!


to = 2026-08-04 01:36:31.311678


,stid,loc,channel,last_connection,delay1_seconds,delay2_seconds,status
15,AGCC,00,HHE,2026-08-03 05:05:29.970000,7.385621e+04,7.386134e+04,Offline
16,AGCC,00,HHN,2026-08-03 05:05:30.470000,7.385571e+04,7.386084e+04,Offline
17,AGCC,00,HHZ,2026-08-03 05:05:29.750000,7.385643e+04,7.386156e+04,Offline
27,ARAMC,00,HHE,2026-08-04 01:36:30,4.420885e+00,1.311678e+00,Online
28,ARAMC,00,HHN,2026-08-04 01:36:30.010000,4.410885e+00,1.301678e+00,Online
29,ARAMC,00,HHZ,2026-08-04 01:36:28.240000,2.060885e+00,3.071678e+00,Online
75,BRJC,00,HHE,2026-08-03 21:08:17.640000,1.608854e+04,1.609367e+04,Offline
76,BRJC,00,HHN,2026-08-03 21:08:17.950000,1.608823e+04,1.609336e+04,Offline
77,BRJC,00,HHZ,2026-08-03 21:08:16.360000,1.608982e+04,1.609495e+04,Offline
78,BRJC,10,HNE,2026-08-03 21:08:17.730000,1.608845e+04,1.609358e+04,Offline


Analyzing results, we note an inconsistency in status for CHI, DBB, GUY2C stations. Status is assigned to intermitent for some sensors (particularly broadband and short-period), but the acelerographs are online. Additionaly, when we look closely his diference in delay times are a little bit higher than 5.1 seconds.  

#### Time difference = 10 s

In [ ]:
# Request slinktool data 
port = SlinktoolExecutor.get_port('91')
comm = SlinktoolExecutor.build_command('91', port)
output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data = DelayCalculator.add_stations_untracked(all_data, slink_data)

# Compute Delay Time
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data = DelayCalculator.compute_delays(final_data, actual_time, 'last_connection')
final_data = DelayCalculator.format_delays(final_data, delay_column='delay')
print(f'to = {actual_time}')

# Second request
period_time = 10
with tqdm(total=period_time, desc=f"Waiting {period_time} seconds to execute a new validation") as pbar:
        for _ in range(period_time):
                time.sleep(1)
                pbar.update(1)

output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data2 = DelayCalculator.add_stations_untracked(all_data, slink_data)
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data2 = DelayCalculator.compute_delays(final_data2, actual_time, 'last_connection')
final_data2 = DelayCalculator.format_delays(final_data2, delay_column='delay')

# Extract delay columns
delay1 = final_data['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
delay2 = final_data2['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
status = StationAnalysis.get_station_status(delay1, delay2, period_time)
final_data2['delay1_seconds'] = delay1
final_data2['delay2_seconds'] = delay2
final_data2['status'] = status
print(f'to = {actual_time}')

# Subset stations of interest 
interm_sts = ['BRJC', 'POP2', 'AGCC', 'PPLP']
perma_sts = ['ARAMC', 'CHI', 'DBB', 'GUY2C']
off_sts = ['MEDEC', 'MOCC', 'GMALF', 'UPD2']
cols = ['stid', 'loc', 'channel', 'last_connection', 'delay1_seconds', 'delay2_seconds', 'status']
res = final_data2.loc[final_data2['stid'].isin([*perma_sts, *interm_sts, *off_sts])]
res = res[cols]
res.head(50)

20:51:33 [INFO] Port stablished succesfully.
20:51:33 [INFO] Sucessful information request from slinktool.
20:51:33 [INFO] Number of sensors to be processed: 1518
20:51:33 [INFO] Success!!
20:51:33 [INFO] Dataframe created succesfully!!
20:51:33 [INFO] Selection of the 792 field sensors completed succesfully!!
20:51:33 [INFO] Transmission type assigned succesfully!!
20:51:33 [INFO] Dataframe formatted succesfully according to SGC data!!
20:51:33 [INFO] Adding stations not tracked by slinktool succesfully!!
20:51:33 [INFO] Delays succesfully computed!!
20:51:33 [INFO] Delays succesfully formatted!!


to = 2026-08-04 01:51:33.568886


Waiting 10 seconds to execute a new validation: 100%|██████████| 10/10 [00:10<00:00,  1.00s/it]
20:51:43 [INFO] Sucessful information request from slinktool.
20:51:43 [INFO] Number of sensors to be processed: 1518
20:51:43 [INFO] Success!!
20:51:43 [INFO] Dataframe created succesfully!!
20:51:43 [INFO] Selection of the 792 field sensors completed succesfully!!
20:51:43 [INFO] Transmission type assigned succesfully!!
20:51:43 [INFO] Dataframe formatted succesfully according to SGC data!!
20:51:43 [INFO] Adding stations not tracked by slinktool succesfully!!
20:51:43 [INFO] Delays succesfully computed!!
20:51:43 [INFO] Delays succesfully formatted!!


to = 2026-08-04 01:51:43.715903


,stid,loc,channel,last_connection,delay1_seconds,delay2_seconds,status
15,AGCC,00,HHE,2026-08-03 05:05:29.970000,7.476360e+04,7.477375e+04,Offline
16,AGCC,00,HHN,2026-08-03 05:05:30.470000,7.476310e+04,7.477325e+04,Offline
17,AGCC,00,HHZ,2026-08-03 05:05:29.750000,7.476382e+04,7.477397e+04,Offline
27,ARAMC,00,HHE,2026-08-04 01:51:40.360000,1.448886e+00,3.355903e+00,Online
28,ARAMC,00,HHN,2026-08-04 01:51:40.510000,1.298886e+00,3.205903e+00,Online
29,ARAMC,00,HHZ,2026-08-04 01:51:42.740000,3.188886e+00,9.759030e-01,Online
75,BRJC,00,HHE,2026-08-03 21:08:17.640000,1.699593e+04,1.700608e+04,Offline
76,BRJC,00,HHN,2026-08-03 21:08:17.950000,1.699562e+04,1.700577e+04,Offline
77,BRJC,00,HHZ,2026-08-03 21:08:16.360000,1.699721e+04,1.700736e+04,Offline
78,BRJC,10,HNE,2026-08-03 21:08:17.730000,1.699584e+04,1.700599e+04,Offline


Analyzing results, we note an inconsistency in status for CHI station. Status is assigned to intermitent in broadband sensor, but the acelerographs are online. Additionaly, when we look closely his diference in delay times are a little bit higher than 10 seconds. 

### Time difference =  15 s

In [ ]:
# Request slinktool data 
port = SlinktoolExecutor.get_port('91')
comm = SlinktoolExecutor.build_command('91', port)
output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data = DelayCalculator.add_stations_untracked(all_data, slink_data)

# Compute Delay Time
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data = DelayCalculator.compute_delays(final_data, actual_time, 'last_connection')
final_data = DelayCalculator.format_delays(final_data, delay_column='delay')
print(f'to = {actual_time}')

# Second request
period_time = 15
with tqdm(total=period_time, desc=f"Waiting {period_time} seconds to execute a new validation") as pbar:
        for _ in range(period_time):
                time.sleep(1)
                pbar.update(1)

output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data2 = DelayCalculator.add_stations_untracked(all_data, slink_data)
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data2 = DelayCalculator.compute_delays(final_data2, actual_time, 'last_connection')
final_data2 = DelayCalculator.format_delays(final_data2, delay_column='delay')

# Extract delay columns
delay1 = final_data['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
delay2 = final_data2['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
status = StationAnalysis.get_station_status(delay1, delay2, period_time)
final_data2['delay1_seconds'] = delay1
final_data2['delay2_seconds'] = delay2
final_data2['status'] = status
print(f'to = {actual_time}')

# Subset stations of interest 
interm_sts = ['BRJC', 'POP2', 'AGCC', 'PPLP']
perma_sts = ['ARAMC', 'CHI', 'DBB', 'GUY2C']
off_sts = ['MEDEC', 'MOCC', 'GMALF', 'UPD2']
cols = ['stid', 'loc', 'channel', 'last_connection', 'delay1_seconds', 'delay2_seconds', 'status']
res = final_data2.loc[final_data2['stid'].isin([*perma_sts, *interm_sts, *off_sts])]
res = res[cols]
res.head(50)

20:53:01 [INFO] Port stablished succesfully.
20:53:01 [INFO] Sucessful information request from slinktool.
20:53:01 [INFO] Number of sensors to be processed: 1518
20:53:01 [INFO] Success!!
20:53:01 [INFO] Dataframe created succesfully!!
20:53:01 [INFO] Selection of the 792 field sensors completed succesfully!!
20:53:01 [INFO] Transmission type assigned succesfully!!
20:53:01 [INFO] Dataframe formatted succesfully according to SGC data!!
20:53:01 [INFO] Adding stations not tracked by slinktool succesfully!!
20:53:01 [INFO] Delays succesfully computed!!
20:53:01 [INFO] Delays succesfully formatted!!


to = 2026-08-04 01:53:01.395213


Waiting 15 seconds to execute a new validation: 100%|██████████| 15/15 [00:15<00:00,  1.00s/it]
20:53:16 [INFO] Sucessful information request from slinktool.
20:53:16 [INFO] Number of sensors to be processed: 1518
20:53:16 [INFO] Success!!
20:53:16 [INFO] Dataframe created succesfully!!
20:53:16 [INFO] Selection of the 792 field sensors completed succesfully!!
20:53:16 [INFO] Transmission type assigned succesfully!!
20:53:16 [INFO] Dataframe formatted succesfully according to SGC data!!
20:53:16 [INFO] Adding stations not tracked by slinktool succesfully!!
20:53:16 [INFO] Delays succesfully computed!!
20:53:16 [INFO] Delays succesfully formatted!!


to = 2026-08-04 01:53:16.545981


,stid,loc,channel,last_connection,delay1_seconds,delay2_seconds,status
15,AGCC,00,HHE,2026-08-03 05:05:29.970000,7.485143e+04,7.486658e+04,Offline
16,AGCC,00,HHN,2026-08-03 05:05:30.470000,7.485093e+04,7.486608e+04,Offline
17,AGCC,00,HHZ,2026-08-03 05:05:29.750000,7.485165e+04,7.486680e+04,Offline
27,ARAMC,00,HHE,2026-08-04 01:53:15.120000,2.755213e+00,1.425981e+00,Online
28,ARAMC,00,HHN,2026-08-04 01:53:15.270000,2.605213e+00,1.275981e+00,Online
29,ARAMC,00,HHZ,2026-08-04 01:53:13.380000,4.495213e+00,3.165981e+00,Online
75,BRJC,00,HHE,2026-08-03 21:08:17.640000,1.708376e+04,1.709891e+04,Offline
76,BRJC,00,HHN,2026-08-03 21:08:17.950000,1.708345e+04,1.709860e+04,Offline
77,BRJC,00,HHZ,2026-08-03 21:08:16.360000,1.708504e+04,1.710019e+04,Offline
78,BRJC,10,HNE,2026-08-03 21:08:17.730000,1.708367e+04,1.709882e+04,Offline


Analyzing results, we don't find an inconsistency in station status and everything look quite consistent between sensors of the same station.

#### Time difference = 30 s

In [ ]:
# Request slinktool data 
port = SlinktoolExecutor.get_port('91')
comm = SlinktoolExecutor.build_command('91', port)
output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data = DelayCalculator.add_stations_untracked(all_data, slink_data)

# Compute Delay Time
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data = DelayCalculator.compute_delays(final_data, actual_time, 'last_connection')
final_data = DelayCalculator.format_delays(final_data, delay_column='delay')
print(f'to = {actual_time}')

# Second request
period_time = 30
with tqdm(total=period_time, desc=f"Waiting {period_time} seconds to execute a new validation") as pbar:
        for _ in range(period_time):
                time.sleep(1)
                pbar.update(1)

output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data2 = DelayCalculator.add_stations_untracked(all_data, slink_data)
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data2 = DelayCalculator.compute_delays(final_data2, actual_time, 'last_connection')
final_data2 = DelayCalculator.format_delays(final_data2, delay_column='delay')

# Extract delay columns
delay1 = final_data['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
delay2 = final_data2['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
status = StationAnalysis.get_station_status(delay1, delay2, period_time)
final_data2['delay1_seconds'] = delay1
final_data2['delay2_seconds'] = delay2
final_data2['status'] = status
print(f'to = {actual_time}')

# Subset stations of interest 
interm_sts = ['BRJC', 'POP2', 'AGCC', 'PPLP']
perma_sts = ['ARAMC', 'CHI', 'DBB', 'GUY2C']
off_sts = ['MEDEC', 'MOCC', 'GMALF', 'UPD2']
cols = ['stid', 'loc', 'channel', 'last_connection', 'delay1_seconds', 'delay2_seconds', 'status']
res = final_data2.loc[final_data2['stid'].isin([*perma_sts, *interm_sts, *off_sts])]
res = res[cols]
res.head(50)

20:56:17 [INFO] Port stablished succesfully.
20:56:17 [INFO] Sucessful information request from slinktool.
20:56:17 [INFO] Number of sensors to be processed: 1518
20:56:17 [INFO] Success!!
20:56:17 [INFO] Dataframe created succesfully!!
20:56:17 [INFO] Selection of the 792 field sensors completed succesfully!!
20:56:17 [INFO] Transmission type assigned succesfully!!
20:56:17 [INFO] Dataframe formatted succesfully according to SGC data!!
20:56:17 [INFO] Adding stations not tracked by slinktool succesfully!!
20:56:17 [INFO] Delays succesfully computed!!
20:56:17 [INFO] Delays succesfully formatted!!


to = 2026-08-04 01:56:17.525440


Waiting 30 seconds to execute a new validation: 100%|██████████| 30/30 [00:30<00:00,  1.00s/it]
20:56:47 [INFO] Sucessful information request from slinktool.
20:56:47 [INFO] Number of sensors to be processed: 1517
20:56:47 [INFO] Success!!
20:56:47 [INFO] Dataframe created succesfully!!
20:56:47 [INFO] Selection of the 792 field sensors completed succesfully!!
20:56:47 [INFO] Transmission type assigned succesfully!!
20:56:47 [INFO] Dataframe formatted succesfully according to SGC data!!
20:56:47 [INFO] Adding stations not tracked by slinktool succesfully!!
20:56:47 [INFO] Delays succesfully computed!!
20:56:47 [INFO] Delays succesfully formatted!!


to = 2026-08-04 01:56:47.700679


,stid,loc,channel,last_connection,delay1_seconds,delay2_seconds,status
15,AGCC,00,HHE,2026-08-03 05:05:29.970000,7.504756e+04,7.507773e+04,Offline
16,AGCC,00,HHN,2026-08-03 05:05:30.470000,7.504706e+04,7.507723e+04,Offline
17,AGCC,00,HHZ,2026-08-03 05:05:29.750000,7.504778e+04,7.507795e+04,Offline
27,ARAMC,00,HHE,2026-08-04 01:56:45.240000,1.125440e+00,2.460679e+00,Online
28,ARAMC,00,HHN,2026-08-04 01:56:45.390000,9.754400e-01,2.310679e+00,Online
29,ARAMC,00,HHZ,2026-08-04 01:56:43.500000,2.865440e+00,4.200679e+00,Online
75,BRJC,00,HHE,2026-08-03 21:08:17.640000,1.727989e+04,1.731006e+04,Offline
76,BRJC,00,HHN,2026-08-03 21:08:17.950000,1.727958e+04,1.730975e+04,Offline
77,BRJC,00,HHZ,2026-08-03 21:08:16.360000,1.728117e+04,1.731134e+04,Offline
78,BRJC,10,HNE,2026-08-03 21:08:17.730000,1.727980e+04,1.730997e+04,Offline


#### Time difference = 45 s

In [ ]:
# Request slinktool data 
port = SlinktoolExecutor.get_port('91')
comm = SlinktoolExecutor.build_command('91', port)
output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data = DelayCalculator.add_stations_untracked(all_data, slink_data)

# Compute Delay Time
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data = DelayCalculator.compute_delays(final_data, actual_time, 'last_connection')
final_data = DelayCalculator.format_delays(final_data, delay_column='delay')
print(f'to = {actual_time}')

# Second request
period_time = 45
with tqdm(total=period_time, desc=f"Waiting {period_time} seconds to execute a new validation") as pbar:
        for _ in range(period_time):
                time.sleep(1)
                pbar.update(1)

output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data2 = DelayCalculator.add_stations_untracked(all_data, slink_data)
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data2 = DelayCalculator.compute_delays(final_data2, actual_time, 'last_connection')
final_data2 = DelayCalculator.format_delays(final_data2, delay_column='delay')

# Extract delay columns
delay1 = final_data['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
delay2 = final_data2['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
status = StationAnalysis.get_station_status(delay1, delay2, period_time)
final_data2['delay1_seconds'] = delay1
final_data2['delay2_seconds'] = delay2
final_data2['status'] = status
print(f'to = {actual_time}')

# Subset stations of interest 
interm_sts = ['BRJC', 'POP2', 'AGCC', 'PPLP']
perma_sts = ['ARAMC', 'CHI', 'DBB', 'GUY2C']
off_sts = ['MEDEC', 'MOCC', 'GMALF', 'UPD2']
cols = ['stid', 'loc', 'channel', 'last_connection', 'delay1_seconds', 'delay2_seconds', 'status']
res = final_data2.loc[final_data2['stid'].isin([*perma_sts, *interm_sts, *off_sts])]
res = res[cols]
res.head(50)

20:58:31 [INFO] Port stablished succesfully.
20:58:32 [INFO] Sucessful information request from slinktool.
20:58:32 [INFO] Number of sensors to be processed: 1517
20:58:32 [INFO] Success!!
20:58:32 [INFO] Dataframe created succesfully!!
20:58:32 [INFO] Selection of the 792 field sensors completed succesfully!!
20:58:32 [INFO] Transmission type assigned succesfully!!
20:58:32 [INFO] Dataframe formatted succesfully according to SGC data!!
20:58:32 [INFO] Adding stations not tracked by slinktool succesfully!!
20:58:32 [INFO] Delays succesfully computed!!
20:58:32 [INFO] Delays succesfully formatted!!


to = 2026-08-04 01:58:32.094884


Waiting 45 seconds to execute a new validation: 100%|██████████| 45/45 [00:45<00:00,  1.00s/it]
20:59:17 [INFO] Sucessful information request from slinktool.
20:59:17 [INFO] Number of sensors to be processed: 1517
20:59:17 [INFO] Success!!
20:59:17 [INFO] Dataframe created succesfully!!
20:59:17 [INFO] Selection of the 792 field sensors completed succesfully!!
20:59:17 [INFO] Transmission type assigned succesfully!!
20:59:17 [INFO] Dataframe formatted succesfully according to SGC data!!
20:59:17 [INFO] Adding stations not tracked by slinktool succesfully!!
20:59:17 [INFO] Delays succesfully computed!!
20:59:17 [INFO] Delays succesfully formatted!!


to = 2026-08-04 01:59:17.304268


,stid,loc,channel,last_connection,delay1_seconds,delay2_seconds,status
15,AGCC,00,HHE,2026-08-03 05:05:29.970000,7.518212e+04,7.522733e+04,Offline
16,AGCC,00,HHN,2026-08-03 05:05:30.470000,7.518162e+04,7.522683e+04,Offline
17,AGCC,00,HHZ,2026-08-03 05:05:29.750000,7.518234e+04,7.522755e+04,Offline
27,ARAMC,00,HHE,2026-08-04 01:59:13.560000,3.854884e+00,3.744268e+00,Online
28,ARAMC,00,HHN,2026-08-04 01:59:13.710000,3.704884e+00,3.594268e+00,Online
29,ARAMC,00,HHZ,2026-08-04 01:59:15.940000,1.474884e+00,1.364268e+00,Online
75,BRJC,00,HHE,2026-08-03 21:08:17.640000,1.741445e+04,1.745966e+04,Offline
76,BRJC,00,HHN,2026-08-03 21:08:17.950000,1.741414e+04,1.745935e+04,Offline
77,BRJC,00,HHZ,2026-08-03 21:08:16.360000,1.741573e+04,1.746094e+04,Offline
78,BRJC,10,HNE,2026-08-03 21:08:17.730000,1.741436e+04,1.745957e+04,Offline


#### Time difference = 60 s

In [ ]:
# Request slinktool data 
port = SlinktoolExecutor.get_port('91')
comm = SlinktoolExecutor.build_command('91', port)
output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data = DelayCalculator.add_stations_untracked(all_data, slink_data)

# Compute Delay Time
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data = DelayCalculator.compute_delays(final_data, actual_time, 'last_connection')
final_data = DelayCalculator.format_delays(final_data, delay_column='delay')
print(f'to = {actual_time}')

# Second request
period_time = 60
with tqdm(total=period_time, desc=f"Waiting {period_time} seconds to execute a new validation") as pbar:
        for _ in range(period_time):
                time.sleep(1)
                pbar.update(1)

output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data2 = DelayCalculator.add_stations_untracked(all_data, slink_data)
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data2 = DelayCalculator.compute_delays(final_data2, actual_time, 'last_connection')
final_data2 = DelayCalculator.format_delays(final_data2, delay_column='delay')

# Extract delay columns
delay1 = final_data['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
delay2 = final_data2['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
status = StationAnalysis.get_station_status(delay1, delay2, period_time)
final_data2['delay1_seconds'] = delay1
final_data2['delay2_seconds'] = delay2
final_data2['status'] = status
print(f'to = {actual_time}')

# Subset stations of interest 
interm_sts = ['BRJC', 'POP2', 'AGCC', 'PPLP']
perma_sts = ['ARAMC', 'CHI', 'DBB', 'GUY2C']
off_sts = ['MEDEC', 'MOCC', 'GMALF', 'UPD2']
cols = ['stid', 'loc', 'channel', 'last_connection', 'delay1_seconds', 'delay2_seconds', 'status']
res = final_data2.loc[final_data2['stid'].isin([*perma_sts, *interm_sts, *off_sts])]
res = res[cols]
res.head(50)

21:01:10 [INFO] Port stablished succesfully.
21:01:10 [INFO] Sucessful information request from slinktool.
21:01:10 [INFO] Number of sensors to be processed: 1517
21:01:10 [INFO] Success!!
21:01:10 [INFO] Dataframe created succesfully!!
21:01:10 [INFO] Selection of the 792 field sensors completed succesfully!!
21:01:10 [INFO] Transmission type assigned succesfully!!
21:01:10 [INFO] Dataframe formatted succesfully according to SGC data!!
21:01:10 [INFO] Adding stations not tracked by slinktool succesfully!!
21:01:10 [INFO] Delays succesfully computed!!
21:01:10 [INFO] Delays succesfully formatted!!


to = 2026-08-04 02:01:10.471070


Waiting 60 seconds to execute a new validation: 100%|██████████| 60/60 [01:00<00:00,  1.00s/it]
21:02:10 [INFO] Sucessful information request from slinktool.
21:02:10 [INFO] Number of sensors to be processed: 1517
21:02:10 [INFO] Success!!
21:02:10 [INFO] Dataframe created succesfully!!
21:02:10 [INFO] Selection of the 792 field sensors completed succesfully!!
21:02:10 [INFO] Transmission type assigned succesfully!!
21:02:10 [INFO] Dataframe formatted succesfully according to SGC data!!
21:02:10 [INFO] Adding stations not tracked by slinktool succesfully!!
21:02:10 [INFO] Delays succesfully computed!!
21:02:10 [INFO] Delays succesfully formatted!!


to = 2026-08-04 02:02:10.713681


,stid,loc,channel,last_connection,delay1_seconds,delay2_seconds,status
15,AGCC,00,HHE,2026-08-03 05:05:29.970000,7.534050e+04,7.540074e+04,Offline
16,AGCC,00,HHN,2026-08-03 05:05:30.470000,7.534000e+04,7.540024e+04,Offline
17,AGCC,00,HHZ,2026-08-03 05:05:29.750000,7.534072e+04,7.540096e+04,Offline
27,ARAMC,00,HHE,2026-08-04 02:02:06.600000,1.551070e+00,4.113681e+00,Online
28,ARAMC,00,HHN,2026-08-04 02:02:06.750000,1.401070e+00,3.963681e+00,Online
29,ARAMC,00,HHZ,2026-08-04 02:02:08.980000,3.291070e+00,1.733681e+00,Online
75,BRJC,00,HHE,2026-08-03 21:08:17.640000,1.757283e+04,1.763307e+04,Offline
76,BRJC,00,HHN,2026-08-03 21:08:17.950000,1.757252e+04,1.763276e+04,Offline
77,BRJC,00,HHZ,2026-08-03 21:08:16.360000,1.757411e+04,1.763435e+04,Offline
78,BRJC,10,HNE,2026-08-03 21:08:17.730000,1.757274e+04,1.763298e+04,Offline


#### Time difference = 90 s

In [ ]:
# Request slinktool data 
port = SlinktoolExecutor.get_port('91')
comm = SlinktoolExecutor.build_command('91', port)
output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data = DelayCalculator.add_stations_untracked(all_data, slink_data)

# Compute Delay Time
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data = DelayCalculator.compute_delays(final_data, actual_time, 'last_connection')
final_data = DelayCalculator.format_delays(final_data, delay_column='delay')
print(f'to = {actual_time}')

# Second request
period_time = 90
with tqdm(total=period_time, desc=f"Waiting {period_time} seconds to execute a new validation") as pbar:
        for _ in range(period_time):
                time.sleep(1)
                pbar.update(1)

output = SlinktoolExecutor.request_info(comm)
solution = SlinktoolFormatter.extract_info(output)
slink_data = SlinktoolFormatter.build_dataframe(solution)
slink_data = SlinktoolFormatter.select_interest_channels(slink_data)
slink_data = SlinktoolFormatter.assign_transmission(slink_data, all_data)
slink_data = SlinktoolFormatter.order_cols(slink_data)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)
    final_data2 = DelayCalculator.add_stations_untracked(all_data, slink_data)
actual_time = datetime.now(UTC).replace(tzinfo=None)
final_data2 = DelayCalculator.compute_delays(final_data2, actual_time, 'last_connection')
final_data2 = DelayCalculator.format_delays(final_data2, delay_column='delay')

# Extract delay columns
delay1 = final_data['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
delay2 = final_data2['delay'].dt.total_seconds().fillna(np.nan).to_numpy()
status = StationAnalysis.get_station_status(delay1, delay2, period_time)
final_data2['delay1_seconds'] = delay1
final_data2['delay2_seconds'] = delay2
final_data2['status'] = status
print(f'to = {actual_time}')

# Subset stations of interest 
interm_sts = ['BRJC', 'POP2', 'AGCC', 'PPLP']
perma_sts = ['ARAMC', 'CHI', 'DBB', 'GUY2C']
off_sts = ['MEDEC', 'MOCC', 'GMALF', 'UPD2']
cols = ['stid', 'loc', 'channel', 'last_connection', 'delay1_seconds', 'delay2_seconds', 'status']
res = final_data2.loc[final_data2['stid'].isin([*perma_sts, *interm_sts, *off_sts])]
res = res[cols]
res.head(50)

21:02:58 [INFO] Port stablished succesfully.
21:02:58 [INFO] Sucessful information request from slinktool.
21:02:58 [INFO] Number of sensors to be processed: 1517
21:02:58 [INFO] Success!!
21:02:58 [INFO] Dataframe created succesfully!!
21:02:58 [INFO] Selection of the 792 field sensors completed succesfully!!
21:02:58 [INFO] Transmission type assigned succesfully!!
21:02:58 [INFO] Dataframe formatted succesfully according to SGC data!!
21:02:58 [INFO] Adding stations not tracked by slinktool succesfully!!
21:02:58 [INFO] Delays succesfully computed!!
21:02:58 [INFO] Delays succesfully formatted!!


to = 2026-08-04 02:02:58.978779


Waiting 90 seconds to execute a new validation: 100%|██████████| 90/90 [01:30<00:00,  1.00s/it]
21:04:29 [INFO] Sucessful information request from slinktool.
21:04:29 [INFO] Number of sensors to be processed: 1518
21:04:29 [INFO] Success!!
21:04:29 [INFO] Dataframe created succesfully!!
21:04:29 [INFO] Selection of the 792 field sensors completed succesfully!!
21:04:29 [INFO] Transmission type assigned succesfully!!
21:04:29 [INFO] Dataframe formatted succesfully according to SGC data!!
21:04:29 [INFO] Adding stations not tracked by slinktool succesfully!!
21:04:29 [INFO] Delays succesfully computed!!
21:04:29 [INFO] Delays succesfully formatted!!


to = 2026-08-04 02:04:29.283584


,stid,loc,channel,last_connection,delay1_seconds,delay2_seconds,status
15,AGCC,00,HHE,2026-08-03 05:05:29.970000,7.544901e+04,7.553931e+04,Offline
16,AGCC,00,HHN,2026-08-03 05:05:30.470000,7.544851e+04,7.553881e+04,Offline
17,AGCC,00,HHZ,2026-08-03 05:05:29.750000,7.544923e+04,7.553953e+04,Offline
27,ARAMC,00,HHE,2026-08-04 02:04:26.680000,2.938779e+00,2.603584e+00,Online
28,ARAMC,00,HHN,2026-08-04 02:04:26.830000,2.788779e+00,2.453584e+00,Online
29,ARAMC,00,HHZ,2026-08-04 02:04:24.940000,4.678779e+00,4.343584e+00,Online
75,BRJC,00,HHE,2026-08-03 21:08:17.640000,1.768134e+04,1.777164e+04,Offline
76,BRJC,00,HHN,2026-08-03 21:08:17.950000,1.768103e+04,1.777133e+04,Offline
77,BRJC,00,HHZ,2026-08-03 21:08:16.360000,1.768262e+04,1.777292e+04,Offline
78,BRJC,10,HNE,2026-08-03 21:08:17.730000,1.768125e+04,1.777155e+04,Offline
